In [1]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| **Build Model** \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

# Build the Neural Network

Neural networks comprise of layers/modules that perform operations on
data. The [torch.nn](https://pytorch.org/docs/stable/nn.html) namespace
provides all the building blocks you need to build your own neural
network. Every module in PyTorch subclasses the
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
A neural network is a module itself that consists of other modules
(layers). This nested structure allows for building and managing complex
architectures easily.

In the following sections, we\'ll build a neural network to classify
images in the FashionMNIST dataset.


In [2]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Get Device for Training

We want to be able to train our model on an
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.


In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [19]:
# 이미지 전처리
transform = transforms.ToTensor()

# 학습 데이터
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

# 테스트 데이터
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)

print("학습 데이터 개수:", len(training_data))
print("테스트 데이터 개수:", len(test_data))

100%|██████████| 26.4M/26.4M [00:01<00:00, 19.4MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 306kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.61MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 20.2MB/s]

학습 데이터 개수: 60000
테스트 데이터 개수: 10000


In [20]:
batch_size = 64

train_dataloader = DataLoader(
    training_data,
    batch_size=batch_size,
    shuffle=True
)

test_dataloader = DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False
)

# 미니배치 하나 확인
for X, y in train_dataloader:
    print("이미지 크기:", X.shape)
    print("정답 크기:", y.shape)
    break

이미지 크기: torch.Size([64, 1, 28, 28])
정답 크기: torch.Size([64])


# Define the Class

We define our neural network by subclassing `nn.Module`, and initialize
the neural network layers in `__init__`. Every `nn.Module` subclass
implements the operations on input data in the `forward` method.


In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

We create an instance of `NeuralNetwork`, and move it to the `device`,
and print its structure.


In [5]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


To use the model, we pass it the input data. This executes the model\'s
`forward`, along with some [background
operations](https://github.com/pytorch/pytorch/blob/270111b7b611d174967ed204776985cefca9c144/torch/nn/modules/module.py#L866).
Do not call `model.forward()` directly!

Calling the model on the input returns a 2-dimensional tensor with dim=0
corresponding to each output of 10 raw predicted values for each class,
and dim=1 corresponding to the individual values of each output. We get
the prediction probabilities by passing it through an instance of the
`nn.Softmax` module.


loss function, Optimizer 설정

In [21]:
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

모델 학습

In [22]:
def train_loop(dataloader, model, loss_fn, optimizer):

    model.train()

    size = len(dataloader.dataset)

    for batch, (X, y) in enumerate(dataloader):

        X = X.to(device)
        y = y.to(device)

        # ① 순전파
        pred = model(X)

        # ② 손실 계산
        loss = loss_fn(pred, y)

        # ③ 이전 기울기 초기화
        optimizer.zero_grad()

        # ④ 역전파
        loss.backward()

        # ⑤ 가중치와 편향 업데이트
        optimizer.step()

        if batch % 100 == 0:
            loss_value = loss.item()
            current = batch * len(X)

            print(
                f"loss: {loss_value:>7f} "
                f"[{current:>5d}/{size:>5d}]"
            )

In [23]:
def test_loop(dataloader, model, loss_fn):

    model.eval()

    size = len(dataloader.dataset)

    test_loss = 0
    correct = 0

    with torch.no_grad():

        for X, y in dataloader:

            X = X.to(device)
            y = y.to(device)

            pred = model(X)

            test_loss += loss_fn(pred, y).item()

            correct += (
                (pred.argmax(1) == y)
                .type(torch.float)
                .sum()
                .item()
            )

    test_loss /= len(dataloader)
    accuracy = correct / size

    print(f"Test Error:")
    print(f"Accuracy: {100 * accuracy:.2f}%")
    print(f"Avg loss: {test_loss:.6f}")

In [24]:
epochs = 5

for t in range(epochs):

    print(f"\nEpoch {t+1}\n-------------------------------")

    train_loop(
        train_dataloader,
        model,
        loss_fn,
        optimizer
    )

    test_loop(
        test_dataloader,
        model,
        loss_fn
    )

print("학습 완료!")


Epoch 1
-------------------------------
loss: 2.308041 [    0/60000]
loss: 0.465772 [ 6400/60000]
loss: 0.453392 [12800/60000]
loss: 0.301830 [19200/60000]
loss: 0.384134 [25600/60000]
loss: 0.470350 [32000/60000]
loss: 0.479594 [38400/60000]
loss: 0.467736 [44800/60000]
loss: 0.290288 [51200/60000]
loss: 0.494750 [57600/60000]
Test Error:
Accuracy: 84.83%
Avg loss: 0.415421

Epoch 2
-------------------------------
loss: 0.274611 [    0/60000]
loss: 0.341213 [ 6400/60000]
loss: 0.357596 [12800/60000]
loss: 0.315839 [19200/60000]
loss: 0.370431 [25600/60000]
loss: 0.275072 [32000/60000]
loss: 0.563889 [38400/60000]
loss: 0.284699 [44800/60000]
loss: 0.344082 [51200/60000]
loss: 0.355961 [57600/60000]
Test Error:
Accuracy: 86.80%
Avg loss: 0.370208

Epoch 3
-------------------------------
loss: 0.315213 [    0/60000]
loss: 0.481904 [ 6400/60000]
loss: 0.284047 [12800/60000]
loss: 0.411737 [19200/60000]
loss: 0.337090 [25600/60000]
loss: 0.387614 [32000/60000]
loss: 0.491324 [38400/60000

In [25]:
torch.save(
    model.state_dict(),
    "fashion_mnist_model.pth"
)

print("모델 저장 완료!")

모델 저장 완료!


In [6]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([7])


------------------------------------------------------------------------


# Model Layers

Let\'s break down the layers in the FashionMNIST model. To illustrate
it, we will take a sample minibatch of 3 images of size 28x28 and see
what happens to it as we pass it through the network.


In [7]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


# nn.Flatten

We initialize the
[nn.Flatten](https://docs.pytorch.org/docs/stable/generated/torch.nn.modules.flatten.Flatten.html)
layer to convert each 2D 28x28 image into a contiguous array of 784
pixel values ( the minibatch dimension (at dim=0) is maintained).


In [8]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


# nn.Linear

The [linear
layer](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
is a module that applies a linear transformation on the input using its
stored weights and biases.


In [9]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


# nn.ReLU

Non-linear activations are what create the complex mappings between the
model\'s inputs and outputs. They are applied after linear
transformations to introduce *nonlinearity*, helping neural networks
learn a wide variety of phenomena.

In this model, we use
[nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
between our linear layers, but there\'s other activations to introduce
non-linearity in your model.


In [10]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.3573, -0.1682, -0.1475, -0.0611,  0.2003, -0.4970,  0.1770,  0.3280,
          0.6599,  0.4581,  0.0958,  0.1336,  0.0366, -0.1637,  0.0988,  0.1974,
          0.4584,  0.1960, -0.1067, -0.1464],
        [-0.1112, -0.1510, -0.3249,  0.2153, -0.0430, -0.6513,  0.5004,  0.5285,
          0.8708,  0.3501,  0.1414, -0.0529,  0.1309,  0.1136,  0.0487, -0.0379,
          0.1430,  0.4882,  0.2945, -0.0627],
        [ 0.0435, -0.2119, -0.1322,  0.2599, -0.0822, -0.3342,  0.1124,  0.2128,
          0.4875,  0.3724,  0.4454, -0.1149,  0.1919, -0.2361,  0.0402, -0.1384,
         -0.1016,  0.2541,  0.1364, -0.1148]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.3573, 0.0000, 0.0000, 0.0000, 0.2003, 0.0000, 0.1770, 0.3280, 0.6599,
         0.4581, 0.0958, 0.1336, 0.0366, 0.0000, 0.0988, 0.1974, 0.4584, 0.1960,
         0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.2153, 0.0000, 0.0000, 0.5004, 0.5285, 0.8708,
         0.3501, 0.1414, 0.0000, 0.1309, 0.1136, 0.04

# nn.Sequential

[nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
is an ordered container of modules. The data is passed through all the
modules in the same order as defined. You can use sequential containers
to put together a quick network like `seq_modules`.


In [11]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

# nn.Softmax

The last linear layer of the neural network returns
[logits]{.title-ref} - raw values in \[-infty, infty\] - which are
passed to the
[nn.Softmax](https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html)
module. The logits are scaled to values \[0, 1\] representing the
model\'s predicted probabilities for each class. `dim` parameter
indicates the dimension along which the values must sum to 1.


In [12]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

# Model Parameters

Many layers inside a neural network are *parameterized*, i.e. have
associated weights and biases that are optimized during training.
Subclassing `nn.Module` automatically tracks all fields defined inside
your model object, and makes all parameters accessible using your
model\'s `parameters()` or `named_parameters()` methods.

In this example, we iterate over each parameter, and print its size and
a preview of its values.


In [13]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0119, -0.0036, -0.0285,  ..., -0.0332, -0.0110,  0.0072],
        [ 0.0065, -0.0067, -0.0063,  ...,  0.0101, -0.0255,  0.0016]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([0.0324, 0.0118], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0080, -0.0353,  0.0347,  ..., -0.0257, -0.0059, -0.0339],
        [-0.0149, -0.0324,  0.0364,  ...,  0.0271,  0.0153,  0.0148]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | Si

In [26]:
from google.colab import files
uploaded = files.upload()

Saving jacket.jpg to jacket (1).jpg


In [27]:
from PIL import Image
from torchvision import transforms

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor()
])

image = Image.open("jacket.jpg")

input_tensor = transform(image)

input_tensor = input_tensor.unsqueeze(0)

input_tensor = input_tensor.to(device)

print(input_tensor.shape)

torch.Size([1, 1, 28, 28])


In [28]:
model.eval()

with torch.no_grad():
    logits = model(input_tensor)

    probabilities = nn.Softmax(dim=1)(logits)

    predicted_class = probabilities.argmax(dim=1)

classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

print("예측 결과:", classes[predicted_class.item()])
print("예측 확률:", probabilities.max().item())

예측 결과: Bag
예측 확률: 0.9999998807907104


In [29]:
model.eval()

with torch.no_grad():
    logits = model(input_tensor)
    probabilities = nn.Softmax(dim=1)(logits)

classes = [
    "T-shirt/top", "Trouser", "Pullover",
    "Dress", "Coat", "Sandal", "Shirt",
    "Sneaker", "Bag", "Ankle boot"
]

for name, prob in zip(classes, probabilities[0]):
    print(f"{name}: {prob.item()*100:.2f}%")

T-shirt/top: 0.00%
Trouser: 0.00%
Pullover: 0.00%
Dress: 0.00%
Coat: 0.00%
Sandal: 0.00%
Shirt: 0.00%
Sneaker: 0.00%
Bag: 100.00%
Ankle boot: 0.00%


------------------------------------------------------------------------
